In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from enum import IntEnum
from typing import Any, Protocol

import gymnasium as gym
import numpy as np
from gymnasium import spaces

from farkle.config import RuleSet
from farkle.scoring import legal_keeps   # -> {dice_used: (score, kept_counts)}


class Decision(IntEnum):
    ROLL = 0
    BANK = 1


class OpponentPolicy(Protocol):
    def play_turn(
        self, rules: RuleSet, rng: np.random.Generator,
        own_score: int, best_rival_score: int, on_board: bool,
    ) -> int:
        """Play one full turn; return points actually banked (0 on farkle)."""

In [ ]:
class FarkleEnv(gym.Env):
    """Farkle from one seat. Opponent turns are simulated inside `step`.

    Action space is flat: 2 * num_dice actions, decoded as
    (dice_kept, Decision). `dice_kept` selects the *highest-scoring* keep
    that uses exactly that many dice -- lower-scoring keeps of the same
    size are provably dominated, since the turn's future depends only on
    (dice_left, turn_score).
    """

    metadata = {"render_modes": ["human", "ansi"]}

    def __init__(
        self,
        rules: RuleSet | str = "configs/rules/standard_10k.yaml",
        opponent: OpponentPolicy | None = None,
        reward_mode: str = "points",       # "points" | "win" | "both"
        max_turns: int = 500,
        render_mode: str | None = None,
    ):
        self.rules = rules if isinstance(rules, RuleSet) else RuleSet.from_yaml(rules)
        self.opponent = opponent
        self.reward_mode = reward_mode
        self.max_turns = max_turns
        self.render_mode = render_mode

        n = self.rules.num_dice
        self.action_space = spaces.Discrete(2 * n)
        self.observation_space = spaces.Dict({
            "roll":       spaces.MultiDiscrete([n + 1] * 6),
            "dice_left":  spaces.Discrete(n + 1),
            "turn_score": spaces.Box(0.0, 3.0, shape=(1,), dtype=np.float32),
            "my_score":   spaces.Box(0.0, 3.5, shape=(1,), dtype=np.float32),
            "opp_score":  spaces.Box(0.0, 3.5, shape=(1,), dtype=np.float32),
            "on_board":   spaces.Discrete(2),
            "final_round": spaces.Discrete(2),
        })
    def _decode(self, action: int) -> tuple[int, Decision]:
        return action // 2 + 1, Decision(action % 2)

    def _encode(self, dice_kept: int, decision: Decision) -> int:
        return (dice_kept - 1) * 2 + int(decision)

    def action_mask(self) -> np.ndarray:
        """Bool array over action_space. Put this in info['action_mask']."""
        mask = np.zeros(self.action_space.n, dtype=bool)
        for dice_used, (score, _) in self._keeps.items():
            mask[self._encode(dice_used, Decision.ROLL)] = True
            if self._bank_allowed(self._turn_score + score):
                mask[self._encode(dice_used, Decision.BANK)] = True
        return mask

    def _bank_allowed(self, prospective_total: int) -> bool:
        """Banking below the first-bank threshold scores nothing and ends the
        turn -- strictly dominated by rolling, since the turn score at risk is
        already worth zero. Mask it out."""
        if self._on_board:
            return True
        return prospective_total >= self.rules.min_first_bank


    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        self._my_score = 0
        self._opp_score = 0
        self._on_board = False
        self._opp_on_board = False
        self._turn_count = 0
        self._final_round = False
        self._finisher: str | None = None
        self._begin_turn()
        return self._obs(), self._info()

    def step(self, action: int):
        dice_kept, decision = self._decode(action)
        if dice_kept not in self._keeps:
            raise ValueError(f"illegal action {action}: no keep uses {dice_kept} dice")

        score, kept = self._keeps[dice_kept]
        self._turn_score += score
        self._dice_left -= dice_kept
        if self._dice_left == 0 and self.rules.hot_dice:
            self._dice_left = self.rules.num_dice        # hot dice

        banked = 0
        if decision is Decision.BANK:
            banked = self._bank()
        else:
            self._roll()
            if not self._keeps:                          # farkle
                self._turn_score = 0
                self._end_turn()
        ...




        return self._obs(), reward, terminated, truncated, self._info()



    def _roll(self) -> None:
        """Roll self._dice_left dice into self._counts; recompute self._keeps."""
        rolled = np.random.choice(range(1,7),self._dice_left)
        self._counts = np.bincount(rolled - 1, minlength=6)

        

    def _begin_turn(self) -> None:
        """Reset turn score, dice_left = num_dice, roll. NB: the opening roll
        can itself be a farkle -- handle a zero-length turn."""

    def _bank(self) -> int:
        """Add turn score to my_score if allowed; set on_board; return points
        banked. Trigger final round if target crossed."""
        self._my_score += self._turn_score
        return self._turn_score


    def _end_turn(self) -> None:
        """Run opponent turn(s) via self.opponent, advance turn counter,
        resolve final-round bookkeeping, then _begin_turn if game continues."""

    def _reward(self, banked: int, terminated: bool) -> float:
        """points: banked / target_score. win: +1/-1/0 at terminal only."""

    def _obs(self) -> dict[str, Any]: ...

    def _info(self) -> dict[str, Any]:
        return {
            "action_mask": self.action_mask(),
            "turn_score": self._turn_score,
            "my_score": self._my_score,
            "opp_score": self._opp_score,
            "rules_fingerprint": self.rules.fingerprint(),
        }

In [15]:
from utils.config import RuleSet

RULES = RuleSet.from_yaml("configs/KCD_4K.yaml")


def score_counts(counts: tuple[int, ...], rules: RuleSet) -> int:
    """counts[i] = number of dice showing face i+1."""
    total = 0
    remaining = list(counts)

    if "full_straight" in rules.specials and all(c == 1 for c in counts):
        return rules.specials["full_straight"]

    if "low_straight" in rules.specials and all(c >= 1 for c in counts[:-1]):
        if counts[0] == 2:
            return rules.specials["low_straight"] + 100
        else:
            return rules.specials["low_straight"]


    if "high_straight" in rules.specials and all(c >= 1 for c in counts[1:]):
        if counts[4] == 2:
            return rules.specials["high_straight"] + 50
        else:
            return rules.specials["high_straight"]

    for face_idx, count in enumerate(remaining):
        face = face_idx + 1
        if count >= 3:
            total += rules.n_of_a_kind_score(face, count)
        elif face in rules.singles:
            total += rules.singles[face] * count

    return total

1050